# CityFlow Traffic Simulation
### فقط از منوی بالا **Runtime ← Run all** بزنید
---

In [ ]:
# ── مرحله ۱: همگام Git + نصب افزونه (فقط در صورت نیاز) ─────────────
import os, subprocess, hashlib, glob

REPO = '/content/repo'
BRANCH = 'main'

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

def glob_cityflow_so():
    return glob.glob(f'{REPO}/cityflow.cpython-*.so')

def native_build_fingerprint(repo):
    h = hashlib.sha256()
    paths = [f'{repo}/CMakeLists.txt', f'{repo}/install.sh']
    paths += sorted(glob.glob(f'{repo}/src/**/*.cpp', recursive=True))
    paths += sorted(glob.glob(f'{repo}/src/**/CMakeLists.txt', recursive=True))
    for p in paths:
        try:
            with open(p, 'rb') as f:
                h.update(os.path.basename(p).encode())
                h.update(f.read())
        except OSError:
            continue
    return h.hexdigest()

# برای حذف کامل /content/repo و شروع از صفر این را True کنید
FORCE_FULL_REINSTALL = False

if FORCE_FULL_REINSTALL:
    sh(f'rm -rf "{REPO}"')

if not os.path.isdir(os.path.join(REPO, '.git')):
    print('⏳ کلون مخزن...')
    sh(f'git clone https://github.com/persiagfx/cityflow.git "{REPO}" --quiet')
else:
    print('⏳ همگام‌سازی با GitHub (reset به آخرین origin/%s)...' % BRANCH)
    sh(f'git -C "{REPO}" fetch origin')
    sh(f'git -C "{REPO}" checkout {BRANCH}')
    sh(f'git -C "{REPO}" reset --hard origin/{BRANCH}')

fp = native_build_fingerprint(REPO)
marker = os.path.join(REPO, '.colab_native_fingerprint')
so_ok = len(glob_cityflow_so()) > 0

need_build = True
if so_ok and os.path.isfile(marker):
    try:
        with open(marker) as mf:
            if mf.read().strip() == fp:
                need_build = False
    except OSError:
        pass

if need_build:
    print('⏳ کامپایل CityFlow (~۳–۴ دقیقه؛ اولین بار یا بعد از تغییر C++/CMake)...')
    sh(f'bash "{REPO}/install.sh"')
    with open(marker, 'w') as mf:
        mf.write(fp)
else:
    print('✓ افزونهٔ Python همین نسخه قبلاً build شده؛ frontend با Git به‌روز شد.')

print('✓ مرحلهٔ ۱ تمام')

In [ ]:
# ── مرحله ۲: اجرای شبیه‌سازی ──────────────────────────
import sys, os, shutil

sys.path.insert(0, '/content/repo')
os.chdir('/content/repo')

import cityflow

STEPS = 300  # تعداد گام‌های شبیه‌سازی

eng = cityflow.Engine('examples/config.json', thread_num=4)
for i in range(STEPS):
    eng.next_step()

shutil.copy('examples/replay.txt',          'frontend/replay.txt')
shutil.copy('examples/replay_roadnet.json', 'frontend/replay_roadnet.json')

print(f'✓ Simulation done — {STEPS} steps | vehicles: {eng.get_vehicle_count()}')

In [ ]:
# ── مرحله ۳: نمایش — Cloudflare + لینک جایگزین Bore (برای Safari) ─────
import subprocess, threading, time, re, os, socket, platform

PORT = 8765
HOST = '127.0.0.1'
LOG_CF = '/tmp/cityflow_cf.log'
LOG_BO = '/tmp/cityflow_bore.log'
URL_CF_RE = re.compile(r'https://[a-zA-Z0-9_.-]+\.trycloudflare\.com')
LISTEN_RE = re.compile(r'listening at (\S+)', re.I)

def strip_ansi(b):
    return re.sub(rb'\x1b\[[0-9;]*[a-zA-Z]', b'', b).decode('utf-8', errors='ignore')

def wait_http_port(secs=45):
    deadline = time.time() + secs
    while time.time() < deadline:
        try:
            s = socket.create_connection((HOST, PORT), timeout=2)
            s.close()
            return True
        except OSError:
            time.sleep(0.4)
    return False

def run_server():
    subprocess.run(
        [
            'python3', '-m', 'http.server', str(PORT),
            '--bind', HOST,
            '--directory', '/content/repo/frontend',
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

def bore_tarball_url():
    u = platform.machine().lower()
    base = 'https://github.com/ekzhang/bore/releases/download/v0.5.0/'
    if 'aarch64' in u or 'arm64' in u:
        return base + 'bore-v0.5.0-aarch64-unknown-linux-musl.tar.gz'
    return base + 'bore-v0.5.0-x86_64-unknown-linux-musl.tar.gz'

def ensure_bore():
    b = '/content/bore'
    if os.path.isfile(b) and os.access(b, os.X_OK):
        return b
    print('⏳ دانلود bore (تونل جایگزین، ~۱ مگابایت)…')
    subprocess.run(
        ['wget', '-q', bore_tarball_url(), '-O', '/tmp/bore.tgz'],
        check=True,
    )
    subprocess.run(['tar', '-xzf', '/tmp/bore.tgz', '-C', '/content'], check=True)
    if not os.path.isfile(b):
        raise RuntimeError('extract bore failed')
    os.chmod(b, 0o755)
    return b

out = {'cf': None, 'bore': None}

def start_bore_tunnel():
    try:
        bore_bin = ensure_bore()
    except Exception as e:
        print('  (bore نصب نشد:', e, ')')
        return
    try:
        os.remove(LOG_BO)
    except OSError:
        pass
    lf = open(LOG_BO, 'wb', buffering=0)
    subprocess.Popen(
        [bore_bin, 'local', str(PORT), '--to', 'bore.pub'],
        stdout=lf,
        stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL,
    )
    for _ in range(90):
        time.sleep(0.5)
        try:
            raw = open(LOG_BO, 'rb').read()
            t = strip_ansi(raw)
            m = LISTEN_RE.search(t)
            if m:
                hostport = m.group(1).rstrip('.,)')
                if not hostport.startswith('http'):
                    out['bore'] = 'http://' + hostport
                else:
                    out['bore'] = hostport
                break
            m2 = re.search(r'bore\.pub:\d+', t)
            if m2:
                out['bore'] = 'http://' + m2.group(0)
                break
        except OSError:
            pass
    try:
        lf.close()
    except Exception:
        pass

subprocess.run('pkill -f cloudflared 2>/dev/null || true', shell=True)
subprocess.run('pkill -x bore 2>/dev/null || true', shell=True)
time.sleep(0.5)

threading.Thread(target=run_server, daemon=True).start()
print('⏳ منتظر بالا آمدن سرور محلی…')
if not wait_http_port(50):
    print('✗ سرور روی پورت', PORT, 'بالا نیامد — Runtime → Restart runtime')
    raise SystemExit(1)
print('✓ سرور محلی آماده')

threading.Thread(target=start_bore_tunnel, daemon=True).start()

bin_path = '/content/cloudflared'
if not os.path.isfile(bin_path):
    print('⏳ دانلود cloudflared…')
    r = subprocess.run(
        [
            'wget', '-q',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-O', bin_path,
        ],
    )
    if r.returncode != 0:
        print('✗ cloudflared دانلود نشد')
    else:
        os.chmod(bin_path, 0o755)

url_cf = None
if os.path.isfile(bin_path):
    for attempt in range(1, 4):
        print(f'⏳ لینک Cloudflare (تلاش {attempt}/۳، تا ~۱۲۰ث)…')
        subprocess.run('pkill -f cloudflared 2>/dev/null || true', shell=True)
        time.sleep(1)
        try:
            os.remove(LOG_CF)
        except OSError:
            pass
        logf = open(LOG_CF, 'wb', buffering=0)
        proc = subprocess.Popen(
            [
                bin_path, 'tunnel', '--no-autoupdate',
                '--url', f'http://{HOST}:{PORT}',
            ],
            stdout=logf,
            stderr=subprocess.STDOUT,
            stdin=subprocess.DEVNULL,
        )
        for _ in range(120):
            time.sleep(1)
            try:
                blob = open(LOG_CF, 'rb').read()
                m = URL_CF_RE.search(strip_ansi(blob))
                if m:
                    url_cf = m.group(0).rstrip('.,;)')
                    break
            except OSError:
                pass
            if proc.poll() is not None:
                break
        try:
            logf.close()
        except Exception:
            pass
        if url_cf:
            break

for _ in range(40):
    if out['bore']:
        break
    time.sleep(0.5)

url_bore = out['bore']

print()
print('=' * 55)
print('  لینک Cloudflare (HTTPS):', url_cf)
print('  لینک Bore جایگزین (HTTP): ', url_bore)
print('=' * 55)
print()
if url_bore:
    print('  ⭐ اگر Safari می‌گوید «Can\'t Find the Server» برای trycloudflare:')
    print('     از لینک Bore (http://bore.pub:…) استفاده کنید یا مرورگر Chrome.')
if url_cf:
    print('  ① Cloudflare: Roadnet ← replay_roadnet.json ، Replay ← replay.txt ، Start')
if url_bore and not url_cf:
    print('  ① Bore: همان فایل‌ها را لود کنید و Start بزنید.')
if not url_cf and not url_bore:
    print('  ⚠ هیچ تونلی بالا نیامد — Runtime را Restart کنید و سلول ۳ را دوباره اجرا کنید.')
    for name, path in ('cloudflared', LOG_CF), ('bore', LOG_BO):
        print(f'  ── tail {name} ──')
        try:
            print(strip_ansi(open(path, 'rb').read()[-2500:]) or '(خالی)')
        except OSError as e:
            print(e)
print()
print('  💡 Runtime باید روشن بماند؛ لینک‌ها موقت هستند.')

